## Step 1. Imports e contratos

In [0]:
from pyspark.sql import functions as F

from notebooks._shared.contracts import (
    SILVER_CONTRACTS,
    GOLD_CONTRACTS,
)

## Step 2. Cobertura estrutural dos contratos Gold

Compara os campos declarados no contrato Gold com os campos disponíveis nos contratos Silver, distinguindo campos diretamente disponíveis daqueles que exigem derivação.

In [0]:
# mapeia cada campo da Silver para suas tabelas de origem
silver_field_sources = {}
for contract in SILVER_CONTRACTS:
    for field in contract.schema.fields:
        silver_field_sources.setdefault(field.name, []).append(contract.name)

# valida a linhagem da Gold, identifica se a coluna vem da Silver ou se precisa ser calculada
for gold_contract in GOLD_CONTRACTS:
    print(f"\n{gold_contract.name}")

    for field in gold_contract.schema.fields:
        sources = silver_field_sources.get(field.name, [])

        print(
            f"  {field.name:<30} "
            f"{'Silver: ' + ', '.join(sources) if sources else 'COLUMA DERIVADA'}"
        )

### Step 3. Derivações Gold

A inspeção confirma que os campos Gold sem correspondência direta na Silver
podem ser derivados dos dados disponíveis:

- `release_year`: `year(release_date)`;
- `commercial_metrics_eligible`: `budget > 0`;
- `profit`: `revenue - budget` quando elegível;
- `roi`: `(revenue - budget) / budget` quando elegível;
- `participation_type`: identifica a origem `cast` ou `crew`;
- `language_role`: identifica o idioma como `original` ou `spoken`.

Para registros não elegíveis às métricas comerciais, `profit` e `roi`
permanecem nulos.

Nenhum campo derivado proposto para Gold ficou sem origem identificada na Silver.